In [1]:
from dataclasses import dataclass
from typing import Tuple

# Set to True to use the paper's hyperparameters, False for manual input
PAPER_HYPERPARAMETERS = True

@dataclass
class CABConfig:
    # Forecast setup
    h: int = 10
    L: int = 120

    # Assets
    asset_order: Tuple[str, ...] = (
        "US_EQUITY", "INTL_EQUITY", "US_BONDS", "INTL_BONDS",
        "US_REIT", "INTL_REIT", "GOLD", "BTC"
    )

    # Data
    data_dir: str = "../proxy"
    input_lag: int = h

    # Time split
    train_end_date: str = "2020-12-31"
    test_start_date: str = "2021-01-01"
    test_end_date: str = "2025-12-31"

    # Numerical stability
    eps: float = 1e-12

    # Manual training hyperparameters
    batch_size: int = 32
    epochs_initial: int = 100
    epochs_refit: int = 5
    learning_rate: float = 1e-05
    weight_decay: float = 1e-5
    optimizer: str = "Sgd"

    # Manual CAB hyperparameters
    kernel_size: int = 3
    hidden_dim: int = 128
    num_attention_heads: int = 16
    num_lstm_layers: int = 5
    dropout: float = 0.2
    shrinkage_phi: float = 0.8

    # Recalibration
    use_warm_start: bool = True


CFG = CABConfig()

# Apply Paper Hyperparameters if the flag is True
if PAPER_HYPERPARAMETERS:
    CFG.L = 378
    CFG.batch_size = 128
    CFG.epochs_initial = 2
    CFG.learning_rate = 1e-4
    CFG.kernel_size = 5
    CFG.hidden_dim = 128
    CFG.num_lstm_layers = 7
    CFG.num_attention_heads = 16
    CFG.shrinkage_phi = 0.8
    CFG.optimizer = "Adam"

# ---------------------------------------------------------
# Derived settings
# ---------------------------------------------------------
COV_DATA_PATH = f"{CFG.data_dir}/realized_cov_target_h{CFG.h}_lag_5_10_21_63.csv"
N_ASSETS = len(CFG.asset_order)
INPUT_PREFIX = f"lag{CFG.input_lag}"
TARGET_PREFIX = "target"
RUN_NAME = f"cab_h{CFG.h}_L{CFG.L}_lag{CFG.input_lag}"

print("CAB configuration loaded.")
print(f"Using Paper Hyperparameters : {PAPER_HYPERPARAMETERS}")
print("Run name                    :", RUN_NAME)
print("Covariance data file        :", COV_DATA_PATH)
print("Horizon h                   :", CFG.h)
print("Sequence length L           :", CFG.L)
print("Input lag                   :", CFG.input_lag)
print("Kernel size                 :", CFG.kernel_size)
print("Train end                   :", CFG.train_end_date)
print("Test period                 :", CFG.test_start_date, "->", CFG.test_end_date)
print("Assets                      :", N_ASSETS)
print("Asset order                 :", CFG.asset_order)

CAB configuration loaded.
Using Paper Hyperparameters : True
Run name                    : cab_h10_L378_lag10
Covariance data file        : ../proxy/realized_cov_target_h10_lag_5_10_21_63.csv
Horizon h                   : 10
Sequence length L           : 378
Input lag                   : 10
Kernel size                 : 5
Train end                   : 2020-12-31
Test period                 : 2021-01-01 -> 2025-12-31
Assets                      : 8
Asset order                 : ('US_EQUITY', 'INTL_EQUITY', 'US_BONDS', 'INTL_BONDS', 'US_REIT', 'INTL_REIT', 'GOLD', 'BTC')


In [2]:
import torch
import torch.nn as nn

In [3]:
def reshape_for_conv3d(x: torch.Tensor, cfg) -> torch.Tensor:
    """
    Convert CAB input from shape (B, L, N, N) to (B, 1, L, N, N).

    Parameters
    ----------
    x : torch.Tensor
        Input tensor with shape (batch_size, sequence_length, n_assets, n_assets).
    cfg : CABConfig
        Global configuration object.

    Returns
    -------
    torch.Tensor
        Reshaped tensor with shape (batch_size, 1, sequence_length, n_assets, n_assets).
    """
    if x.ndim != 4:
        raise ValueError(f"Expected x.ndim == 4, got {x.ndim}")

    B, L, N1, N2 = x.shape

    if L != cfg.L:
        raise ValueError(f"Expected sequence length {cfg.L}, got {L}")

    if N1 != len(cfg.asset_order) or N2 != len(cfg.asset_order):
        raise ValueError(
            f"Expected matrix size ({len(cfg.asset_order)}, {len(cfg.asset_order)}), "
            f"got ({N1}, {N2})"
        )

    return x.unsqueeze(1)

In [4]:
def flatten_conv_output_for_lstm(x_conv: torch.Tensor) -> torch.Tensor:
    """
    Convert CNN output from shape (B, C, L, N, N) to (B, L, C*N*N).

    Parameters
    ----------
    x_conv : torch.Tensor
        CNN output tensor.

    Returns
    -------
    torch.Tensor
        Flattened sequence tensor suitable for an LSTM.
    """
    if x_conv.ndim != 5:
        raise ValueError(f"Expected x_conv.ndim == 5, got {x_conv.ndim}")

    B, C, L, N1, N2 = x_conv.shape

    # Move time dimension to second axis: (B, L, C, N, N)
    x_conv = x_conv.permute(0, 2, 1, 3, 4).contiguous()

    # Flatten each time step
    x_flat = x_conv.view(B, L, C * N1 * N2)

    return x_flat

In [5]:
class CAB3DCNNBlock(nn.Module):
    """
    First CAB stage: 3D convolution over a sequence of covariance matrices.

    Input
    -----
    x : torch.Tensor
        Shape (B, L, N, N)

    Internal Conv3D input
    ---------------------
        Shape (B, 1, L, N, N)

    Outputs
    -------
    x_conv : torch.Tensor
        Shape (B, 1, L, N, N) if out_channels = 1
    x_flat : torch.Tensor
        Shape (B, L, N*N) if out_channels = 1
    """
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        padding = cfg.kernel_size // 2

        self.conv3d = nn.Conv3d(
            in_channels=1,
            out_channels=1,
            kernel_size=(cfg.kernel_size, cfg.kernel_size, cfg.kernel_size),
            stride=1,
            padding=(padding, padding, padding)
        )

        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor):
        # (B, L, N, N) -> (B, 1, L, N, N)
        x = reshape_for_conv3d(x, self.cfg)

        # 3D convolution
        x_conv = self.conv3d(x)

        # Nonlinearity
        x_conv = self.activation(x_conv)

        # Flatten for later recurrent layer
        x_flat = flatten_conv_output_for_lstm(x_conv)

        return x_conv, x_flat

In [7]:
CNN_FEATURE_DIM = N_ASSETS * N_ASSETS   # valid for out_channels = 1

print("CNN feature dimension :", CNN_FEATURE_DIM)
print("BiLSTM hidden dim     :", CFG.hidden_dim)
print("BiLSTM output dim     :", 2 * CFG.hidden_dim)

CNN feature dimension : 64
BiLSTM hidden dim     : 128
BiLSTM output dim     : 256


In [8]:
def validate_lstm_input(x_seq: torch.Tensor, cfg) -> None:
    """
    Validate that the BiLSTM input has shape (B, L, F).

    Parameters
    ----------
    x_seq : torch.Tensor
        Sequence tensor from the CNN block.
    cfg : CABConfig
        Global configuration object.
    """
    if x_seq.ndim != 3:
        raise ValueError(f"Expected x_seq.ndim == 3, got {x_seq.ndim}")

    B, L, F = x_seq.shape

    if L != cfg.L:
        raise ValueError(f"Expected sequence length {cfg.L}, got {L}")

    if F <= 0:
        raise ValueError(f"Feature dimension must be positive, got {F}")

In [9]:
class CABBiLSTMBlock(nn.Module):
    """
    CAB BiLSTM stage.

    Input
    -----
    x_seq : torch.Tensor
        Shape (B, L, F)

    Output
    ------
    x_lstm : torch.Tensor
        Shape (B, L, 2 * hidden_dim)
    """
    def __init__(self, cfg, input_dim: int):
        super().__init__()
        self.cfg = cfg
        self.input_dim = input_dim

        self.bilstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=cfg.hidden_dim,
            num_layers=cfg.num_lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=cfg.dropout if cfg.num_lstm_layers > 1 else 0.0
        )

        self.output_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x_seq: torch.Tensor) -> torch.Tensor:
        validate_lstm_input(x_seq, self.cfg)

        x_lstm, _ = self.bilstm(x_seq)
        x_lstm = self.output_dropout(x_lstm)

        return x_lstm

In [25]:
def validate_attention_config(cfg) -> None:
    """
    Validate that the attention setup is internally consistent.

    In particular, embed_dim = 2 * hidden_dim must be divisible by the
    number of attention heads.
    """
    embed_dim = 2 * cfg.hidden_dim

    if embed_dim % cfg.num_attention_heads != 0:
        raise ValueError(
            f"embed_dim = 2 * hidden_dim = {embed_dim} must be divisible by "
            f"num_attention_heads = {cfg.num_attention_heads}"
        )

In [10]:
def validate_attention_input(x_seq: torch.Tensor, cfg) -> None:
    """
    Validate that the attention input has shape (B, L, F),
    where F = 2 * hidden_dim.

    Parameters
    ----------
    x_seq : torch.Tensor
        Sequence tensor from the BiLSTM block.
    cfg : CABConfig
        Global configuration object.
    """
    if x_seq.ndim != 3:
        raise ValueError(f"Expected x_seq.ndim == 3, got {x_seq.ndim}")

    B, L, F = x_seq.shape

    if L != cfg.L:
        raise ValueError(f"Expected sequence length {cfg.L}, got {L}")

    expected_F = 2 * cfg.hidden_dim
    if F != expected_F:
        raise ValueError(f"Expected feature dimension {expected_F}, got {F}")

In [11]:
def mean_pool_sequence(x_seq: torch.Tensor) -> torch.Tensor:
    """
    Mean-pool across the sequence dimension.

    Parameters
    ----------
    x_seq : torch.Tensor
        Tensor of shape (B, L, F)

    Returns
    -------
    torch.Tensor
        Tensor of shape (B, F)
    """
    if x_seq.ndim != 3:
        raise ValueError(f"Expected x_seq.ndim == 3, got {x_seq.ndim}")

    return x_seq.mean(dim=1)

In [12]:
class CABAttentionBlock(nn.Module):
    """
    CAB multi-head self-attention block.

    Input
    -----
    x_seq : torch.Tensor
        Shape (B, L, 2 * hidden_dim)

    Outputs
    -------
    x_attn : torch.Tensor
        Attended sequence, shape (B, L, 2 * hidden_dim)

    a_mean : torch.Tensor
        Mean-pooled context vector, shape (B, 2 * hidden_dim)

    attn_weights : torch.Tensor
        Attention weights returned by PyTorch.
    """
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed_dim = 2 * cfg.hidden_dim

        validate_attention_config(cfg)

        self.attention = nn.MultiheadAttention(
            embed_dim=self.embed_dim,
            num_heads=cfg.num_attention_heads,
            dropout=cfg.dropout,
            batch_first=True
        )

        self.output_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x_seq: torch.Tensor):
        validate_attention_input(x_seq, self.cfg)

        x_attn, attn_weights = self.attention(
            query=x_seq,
            key=x_seq,
            value=x_seq
        )

        x_attn = self.output_dropout(x_attn)
        a_mean = mean_pool_sequence(x_attn)

        return x_attn, a_mean, attn_weights

In [13]:
def validate_head_input(a_mean: torch.Tensor, cfg) -> None:
    """
    Validate that the forecast head input has shape (B, 2 * hidden_dim).
    """
    if a_mean.ndim != 2:
        raise ValueError(f"Expected a_mean.ndim == 2, got {a_mean.ndim}")

    B, F = a_mean.shape
    expected_F = 2 * cfg.hidden_dim

    if F != expected_F:
        raise ValueError(f"Expected feature dimension {expected_F}, got {F}")

In [14]:
def flat_to_matrix(y_flat: torch.Tensor, cfg) -> torch.Tensor:
    """
    Convert flat forecast output from shape (B, N*N) to (B, N, N).
    """
    if y_flat.ndim != 2:
        raise ValueError(f"Expected y_flat.ndim == 2, got {y_flat.ndim}")

    B, D = y_flat.shape
    n_assets = len(cfg.asset_order)
    expected_D = n_assets * n_assets

    if D != expected_D:
        raise ValueError(f"Expected flat dimension {expected_D}, got {D}")

    return y_flat.view(B, n_assets, n_assets)

In [15]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler


def enforce_symmetry(Y: torch.Tensor) -> torch.Tensor:
    """
    Enforce symmetry on a batch of square matrices.
    Y: (B, N, N)
    """
    if Y.ndim != 3:
        raise ValueError(f"Expected Y.ndim == 3, got {Y.ndim}")
    return 0.5 * (Y + Y.transpose(-1, -2))

def enforce_psd(Y_sym: torch.Tensor, eps: float = 1e-8, jitter: float = 1e-12) -> torch.Tensor:
    """
    Enforce SPD by clipping negative eigenvalues to eps and adding identity jitter.
    Y_sym: (B, N, N)
    """
    if Y_sym.ndim != 3:
        raise ValueError(f"Expected Y_sym.ndim == 3, got {Y_sym.ndim}")

    eigvals, eigvecs = torch.linalg.eigh(Y_sym)
    eigvals_clipped = torch.clamp(eigvals, min=eps)
    Y_psd = eigvecs @ torch.diag_embed(eigvals_clipped) @ eigvecs.transpose(-1, -2)

    # re-symmetrize for numerical stability
    Y_psd = 0.5 * (Y_psd + Y_psd.transpose(-1, -2))

    # add identity jitter to ensure strict positive definiteness
    Y_psd = Y_psd + torch.eye(Y_psd.shape[-1], dtype=Y_psd.dtype, device=Y_psd.device).unsqueeze(0) * jitter

    return Y_psd


def fit_input_scaler(X_train: np.ndarray) -> StandardScaler:
    """
    Fit StandardScaler on flattened covariance entries from TRAINING INPUT SEQUENCES only.

    X_train shape: (n_samples, L, N, N)
    Scaler is fit on shape: (n_samples * L, N*N)
    """
    n_samples, L, N, _ = X_train.shape
    X_flat = X_train.reshape(n_samples * L, N * N)
    scaler = StandardScaler()
    scaler.fit(X_flat)
    return scaler


def transform_cov_sequences(X: np.ndarray, scaler: StandardScaler) -> np.ndarray:
    """
    Apply scaler entrywise to all covariance matrices in a sequence tensor.
    X shape: (n_samples, L, N, N)
    """
    n_samples, L, N, _ = X.shape
    X_flat = X.reshape(n_samples * L, N * N)
    X_scaled = scaler.transform(X_flat).reshape(n_samples, L, N, N)
    return X_scaled.astype(np.float32)


def scaler_to_matrix_params(scaler: StandardScaler, n_assets: int):
    """
    Convert sklearn scaler params to (N, N) mean/std matrices.
    """
    mean_matrix = scaler.mean_.reshape(n_assets, n_assets).astype(np.float32)
    std_matrix = scaler.scale_.reshape(n_assets, n_assets).astype(np.float32)

    # safety
    std_matrix = np.where(std_matrix < 1e-12, 1.0, std_matrix).astype(np.float32)
    return mean_matrix, std_matrix


class FrobeniusLoss(nn.Module):
    """
    Mean Frobenius distance across the batch.
    Compares final unscaled/PSD/shrunk predictions to unscaled targets.
    """
    def __init__(self):
        super().__init__()

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        diff = y_pred - y_true
        frob = torch.linalg.matrix_norm(diff, ord='fro', dim=(-2, -1))
        return frob.mean()

In [16]:
def inspect_matrix_batch(Y_raw: torch.Tensor, Y_sym: torch.Tensor, Y_psd: torch.Tensor) -> None:
    """
    Print simple diagnostics for a batch of matrices.
    """
    raw_sym_error = torch.max(torch.abs(Y_raw - Y_raw.transpose(-1, -2))).item()
    sym_sym_error = torch.max(torch.abs(Y_sym - Y_sym.transpose(-1, -2))).item()
    psd_sym_error = torch.max(torch.abs(Y_psd - Y_psd.transpose(-1, -2))).item()

    min_eig_sym = torch.min(torch.linalg.eigvalsh(Y_sym)).item()
    min_eig_psd = torch.min(torch.linalg.eigvalsh(Y_psd)).item()

    print("Max asymmetry raw :", raw_sym_error)
    print("Max asymmetry sym :", sym_sym_error)
    print("Max asymmetry psd :", psd_sym_error)
    print("Min eigenvalue sym:", min_eig_sym)
    print("Min eigenvalue psd:", min_eig_psd)

In [17]:
class CABForecastHead(nn.Module):
    """
    Final CAB forecast head.

    Input
    -----
    a_mean : torch.Tensor
        Shape (B, 2 * hidden_dim)

    Outputs
    -------
    y_flat : torch.Tensor
        Shape (B, N*N)

    Y_raw : torch.Tensor
        Shape (B, N, N)
    """
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.n_assets = len(cfg.asset_order)
        self.output_dim = self.n_assets * self.n_assets

        self.fc = nn.Linear(2 * cfg.hidden_dim, self.output_dim)

    def forward(self, a_mean: torch.Tensor):
        validate_head_input(a_mean, self.cfg)

        y_flat = self.fc(a_mean)
        Y_raw = flat_to_matrix(y_flat, self.cfg)

        return y_flat, Y_raw

In [18]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm

# =========================================================
# DEVICE
# =========================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# =========================================================
# FULL CAB MODEL
# Assumes these already exist from earlier cells:
# - CAB3DCNNBlock
# - CABBiLSTMBlock
# - CABAttentionBlock
# - CABForecastHead
# - enforce_symmetry
# =========================================================
import torch
import torch.nn as nn


class CABModel(nn.Module):
    """
    Full CAB model with exact post-processing order:

    1. Raw network output in scaled space
    2. Symmetrize
    3. Unscale
    4. PSD correction
    5. Shrinkage with most recent unscaled historical covariance
    """
    def __init__(self, cfg, scaler_mean_matrix, scaler_std_matrix):
        super().__init__()
        self.cfg = cfg
        self.n_assets = len(cfg.asset_order)

        self.cnn_block = CAB3DCNNBlock(cfg)

        cnn_feature_dim = self.n_assets * self.n_assets

        self.bilstm_block = CABBiLSTMBlock(
            cfg=cfg,
            input_dim=cnn_feature_dim
        )

        self.attention_block = CABAttentionBlock(cfg)
        self.forecast_head = CABForecastHead(cfg)

        # store scaler params as buffers so they move with model.to(device)
        self.register_buffer(
            "scaler_mean_matrix",
            torch.tensor(scaler_mean_matrix, dtype=torch.float32)
        )
        self.register_buffer(
            "scaler_std_matrix",
            torch.tensor(scaler_std_matrix, dtype=torch.float32)
        )

    def unscale_matrix(self, Y_scaled: torch.Tensor) -> torch.Tensor:
        """
        Reverse StandardScaler entrywise.
        Y_scaled: (B, N, N)
        """
        return Y_scaled * self.scaler_std_matrix.unsqueeze(0) + self.scaler_mean_matrix.unsqueeze(0)

    def forward(self, x: torch.Tensor):
        """
        x is expected to be SCALED input sequences of shape (B, L, N, N).
        """
        # save the most recent historical covariance from the input sequence
        # x[:, -1] is in scaled space, so unscale it before shrinkage
        D_t_scaled = x[:, -1, :, :]                 # (B, N, N)
        D_t_unscaled = self.unscale_matrix(D_t_scaled)

        x_conv, x_flat = self.cnn_block(x)
        x_lstm = self.bilstm_block(x_flat)
        x_attn, a_mean, attn_weights = self.attention_block(x_lstm)
        y_flat, y_raw_scaled = self.forecast_head(a_mean)

        # exact paper-style post-processing order
        y_sym_scaled = enforce_symmetry(y_raw_scaled)
        y_sym_unscaled = self.unscale_matrix(y_sym_scaled)
        y_psd = enforce_psd(y_sym_unscaled, eps=0.0)

        phi = self.cfg.shrinkage_phi
        y_final = phi * y_psd + (1.0 - phi) * D_t_unscaled
        y_final = enforce_symmetry(y_final)

        return {
            "x_conv": x_conv,
            "x_flat": x_flat,
            "x_lstm": x_lstm,
            "x_attn": x_attn,
            "a_mean": a_mean,
            "attn_weights": attn_weights,
            "y_flat": y_flat,
            "y_raw_scaled": y_raw_scaled,
            "y_sym_scaled": y_sym_scaled,
            "y_sym_unscaled": y_sym_unscaled,
            "y_psd": y_psd,
            "D_t_unscaled": D_t_unscaled,
            "y_final": y_final,
        }





def make_dataloader(X, Y, batch_size, shuffle=True):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    Y_tensor = torch.tensor(Y, dtype=torch.float32)

    dataset = TensorDataset(X_tensor, Y_tensor)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False
    )
    return loader


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0
    n_samples = 0

    pbar = tqdm(loader, desc="Training", leave=False)

    for x_batch, y_batch in pbar:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)   # keep targets unscaled

        optimizer.zero_grad()

        out = model(x_batch)
        y_pred = out["y_final"]        # final unscaled + PSD + shrinkage output

        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()

        batch_size = x_batch.size(0)
        running_loss += loss.item() * batch_size
        n_samples += batch_size

        pbar.set_postfix({"batch_loss": f"{loss.item():.6f}"})

    epoch_loss = running_loss / n_samples
    return epoch_loss


@torch.no_grad()
def evaluate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    n_samples = 0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        out = model(x_batch)
        y_pred = out["y_final"]

        loss = criterion(y_pred, y_batch)

        batch_size = x_batch.size(0)
        running_loss += loss.item() * batch_size
        n_samples += batch_size

    return running_loss / n_samples


def fit_cab_model(X_train_scaled, Y_train_unscaled, cfg, device, scaler_mean_matrix, scaler_std_matrix,
                  X_val_scaled=None, Y_val_unscaled=None):
    model = CABModel(
        cfg=cfg,
        scaler_mean_matrix=scaler_mean_matrix,
        scaler_std_matrix=scaler_std_matrix
    ).to(device)

    train_loader = make_dataloader(
        X=X_train_scaled,
        Y=Y_train_unscaled,
        batch_size=cfg.batch_size,
        shuffle=True
    )

    val_loader = None
    if X_val_scaled is not None and Y_val_unscaled is not None:
        val_loader = make_dataloader(
            X=X_val_scaled,
            Y=Y_val_unscaled,
            batch_size=cfg.batch_size,
            shuffle=False
        )

    criterion = FrobeniusLoss()

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay
    )

    history = {
        "train_loss": [],
        "val_loss": []
    }

    print("Training samples :", len(X_train_scaled))
    print("Batch size       :", cfg.batch_size)
    print("Epochs           :", cfg.epochs_initial)

    for epoch in range(1, cfg.epochs_initial + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device
        )

        history["train_loss"].append(train_loss)

        if val_loader is not None:
            val_loss = evaluate_one_epoch(
                model=model,
                loader=val_loader,
                criterion=criterion,
                device=device
            )
            history["val_loss"].append(val_loss)
            print(
                f"Epoch {epoch:03d}/{cfg.epochs_initial:03d} | "
                f"train_frob = {train_loss:.6f} | val_frob = {val_loss:.6f}"
            )
        else:
            print(
                f"Epoch {epoch:03d}/{cfg.epochs_initial:03d} | "
                f"train_frob = {train_loss:.6f}"
            )

    return model, history

Using device: cpu


In [19]:
import numpy as np
import pandas as pd

# 1) Load covariance dataset
df_cov = pd.read_csv(
    COV_DATA_PATH,
    parse_dates=["Date", "target_start", "target_end"]
).sort_values("Date").reset_index(drop=True)

print("Loaded covariance data:", COV_DATA_PATH)
print("Rows:", len(df_cov))
print("Date range:", df_cov["Date"].min().date(), "->", df_cov["Date"].max().date())


def row_to_cov_matrix(row: pd.Series, prefix: str, asset_order) -> np.ndarray:
    n = len(asset_order)
    M = np.zeros((n, n), dtype=float)

    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            col = f"{prefix}_cov_{a}__{b}"
            if col not in row.index:
                raise ValueError(f"Missing column: {col}")
            M[i, j] = row[col]

    return M


# 2) Build matrix streams
input_prefix = f"lag{CFG.input_lag}"
target_prefix = "target"

D_list = []
S_list = []

for _, row in df_cov.iterrows():
    D_list.append(
        row_to_cov_matrix(row, prefix=input_prefix, asset_order=CFG.asset_order)
    )
    S_list.append(
        row_to_cov_matrix(row, prefix=target_prefix, asset_order=CFG.asset_order)
    )

D_all = np.asarray(D_list, dtype=np.float32)
S_all = np.asarray(S_list, dtype=np.float32)

print("D_all shape:", D_all.shape)
print("S_all shape:", S_all.shape)


# 3) Build sliding-window samples
X_list = []
Y_list = []
sample_dates = []
sample_target_end = []

for end_idx in range(CFG.L - 1, len(df_cov)):
    X_seq = D_all[end_idx - CFG.L + 1 : end_idx + 1]   # (L, N, N)
    Y_mat = S_all[end_idx]                             # (N, N)

    X_list.append(X_seq)
    Y_list.append(Y_mat)
    sample_dates.append(df_cov.loc[end_idx, "Date"])
    sample_target_end.append(df_cov.loc[end_idx, "target_end"])

X_all = np.asarray(X_list, dtype=np.float32)
Y_all = np.asarray(Y_list, dtype=np.float32)
sample_dates = pd.to_datetime(pd.Series(sample_dates))
sample_target_end = pd.to_datetime(pd.Series(sample_target_end))

print("X_all shape:", X_all.shape)
print("Y_all shape:", Y_all.shape)


# 4) Training mask
train_mask = sample_target_end <= pd.Timestamp(CFG.train_end_date)

X_train_full = X_all[train_mask.to_numpy()]
Y_train_full = Y_all[train_mask.to_numpy()]
dates_train_full = sample_dates[train_mask].reset_index(drop=True)

print("X_train_full shape:", X_train_full.shape)
print("Y_train_full shape:", Y_train_full.shape)

if len(dates_train_full) == 0:
    raise ValueError("No training samples found. Check CFG.L, CFG.h, and CFG.train_end_date.")


# 5) 80/20 split inside training period, as in the paper
n_train_total = len(X_train_full)
n_fit = int(0.8 * n_train_total)

X_fit = X_train_full[:n_fit]
Y_fit = Y_train_full[:n_fit]

X_val = X_train_full[n_fit:]
Y_val = Y_train_full[n_fit:]

dates_fit = dates_train_full.iloc[:n_fit].reset_index(drop=True)
dates_val = dates_train_full.iloc[n_fit:].reset_index(drop=True)

print("X_fit shape:", X_fit.shape)
print("Y_fit shape:", Y_fit.shape)
print("X_val shape:", X_val.shape)
print("Y_val shape:", Y_val.shape)


# 6) Fit scaler on TRAINING INPUT SEQUENCES ONLY
input_scaler = fit_input_scaler(X_fit)

n_assets = len(CFG.asset_order)
scaler_mean_matrix, scaler_std_matrix = scaler_to_matrix_params(
    input_scaler,
    n_assets=n_assets
)

print("Scaler mean matrix shape:", scaler_mean_matrix.shape)
print("Scaler std matrix shape :", scaler_std_matrix.shape)


# 7) Transform inputs
X_fit_scaled = transform_cov_sequences(X_fit, input_scaler)
X_val_scaled = transform_cov_sequences(X_val, input_scaler)

# keep targets unscale
Y_fit_unscaled = Y_fit.astype(np.float32)
Y_val_unscaled = Y_val.astype(np.float32)

print("X_fit_scaled shape:", X_fit_scaled.shape)
print("X_val_scaled shape:", X_val_scaled.shape)

Loaded covariance data: ../proxy/realized_cov_target_h10_lag_5_10_21_63.csv
Rows: 2132
Date range: 2017-06-27 -> 2025-12-17
D_all shape: (2132, 8, 8)
S_all shape: (2132, 8, 8)
X_all shape: (1755, 378, 8, 8)
Y_all shape: (1755, 8, 8)
X_train_full shape: (500, 378, 8, 8)
Y_train_full shape: (500, 8, 8)
X_fit shape: (400, 378, 8, 8)
Y_fit shape: (400, 8, 8)
X_val shape: (100, 378, 8, 8)
Y_val shape: (100, 8, 8)
Scaler mean matrix shape: (8, 8)
Scaler std matrix shape : (8, 8)
X_fit_scaled shape: (400, 378, 8, 8)
X_val_scaled shape: (100, 378, 8, 8)


In [ ]:
model, history = fit_cab_model(
    X_train_scaled=X_fit_scaled,
    Y_train_unscaled=Y_fit_unscaled,
    cfg=CFG,
    device=DEVICE,
    scaler_mean_matrix=scaler_mean_matrix,
    scaler_std_matrix=scaler_std_matrix,
    X_val_scaled=X_val_scaled,
    Y_val_unscaled=Y_val_unscaled
)

In [ ]:
# Saving model trained on complete training data, which we later use in the recalibration 

model_path = f"../models/cab_model_h{CFG.h}_new.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "config": CFG,
    "scaler_mean_matrix": scaler_mean_matrix,
    "scaler_std_matrix": scaler_std_matrix
}, model_path)

print("CAB model saved to:")
print(model_path)

Online update of model

In [ ]:
import torch

model_path = f"../models/cab_model_h{CFG.h}_new.pt"

# Allowlist the CABConfig class for safe loading
torch.serialization.add_safe_globals([CABConfig])

checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=False)

base_cfg = checkpoint["config"]
scaler_mean_matrix = checkpoint["scaler_mean_matrix"]
scaler_std_matrix = checkpoint["scaler_std_matrix"]

model_online = CABModel(
    cfg=base_cfg,
    scaler_mean_matrix=scaler_mean_matrix,
    scaler_std_matrix=scaler_std_matrix
).to(DEVICE)

model_online.load_state_dict(checkpoint["model_state_dict"])
model_online.eval()

print("Loaded baseline CAB model from:")
print(model_path)

In [21]:
import numpy as np

def transform_cov_sequences_from_params(X, mean_matrix, std_matrix):
    """
    X: (n_samples, L, N, N)
    """
    mean_matrix = np.asarray(mean_matrix, dtype=np.float32)
    std_matrix = np.asarray(std_matrix, dtype=np.float32)

    X_scaled = (X - mean_matrix[None, None, :, :]) / std_matrix[None, None, :, :]
    return X_scaled.astype(np.float32)

In [22]:
import pandas as pd
import numpy as np

sample_dates_series = pd.Series(sample_dates).reset_index(drop=True)
sample_target_end_series = pd.Series(sample_target_end).reset_index(drop=True)

# last sample included in the initial training set
train_mask_full = sample_target_end_series <= pd.Timestamp(CFG.train_end_date)
train_last_pos = np.where(train_mask_full.to_numpy())[0].max()

print("Last training sample position:", train_last_pos)
print("Last training origin date    :", sample_dates_series.iloc[train_last_pos])
print("Last training target_end     :", sample_target_end_series.iloc[train_last_pos])


test_mask = (
    (sample_dates_series >= pd.Timestamp("2021-01-01")) &
    (sample_dates_series <= pd.Timestamp("2025-12-31"))
)

test_positions = np.where(test_mask.to_numpy())[0]

print("Number of forecast origins:", len(test_positions))
print("First origin:", sample_dates_series.iloc[test_positions[0]])
print("Last  origin:", sample_dates_series.iloc[test_positions[-1]])

Last training sample position: 499
Last training origin date    : 2020-12-17 00:00:00
Last training target_end     : 2020-12-31 00:00:00
Number of forecast origins: 1246
First origin: 2021-01-04 00:00:00
Last  origin: 2025-12-17 00:00:00


In [23]:
import torch
from torch.utils.data import TensorDataset, DataLoader

def online_update_step(
    model,
    X_update_scaled,
    Y_update_unscaled,
    device,
    lr=5e-5,
    epochs=1,
    batch_size=1,
    weight_decay=0.0
):
    """
    Small warm-start fine-tuning step.
    X_update_scaled: (n, L, N, N)
    Y_update_unscaled: (n, N, N)
    """
    model.train()

    X_tensor = torch.tensor(X_update_scaled, dtype=torch.float32)
    Y_tensor = torch.tensor(Y_update_unscaled, dtype=torch.float32)

    loader = DataLoader(
        TensorDataset(X_tensor, Y_tensor),
        batch_size=batch_size,
        shuffle=False,
        drop_last=False
    )

    criterion = FrobeniusLoss()
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    for _ in range(epochs):
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            out = model(x_batch)
            y_pred = out["y_final"]

            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()

    model.eval()
    return model

In [ ]:
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# =========================================================
# ONLINE UPDATE SETTINGS
# =========================================================
ONLINE_LR = 5e-5
ONLINE_EPOCHS = 1         # 1 epoch over the recent window is safer
ONLINE_BATCH_SIZE = 16    # Smooths out the gradient steps
REPLAY_WINDOW = 1        # Train on the last 21 matured days (approx 1 trading month)

pred_list = []
forecast_dates = []
update_log = []

# start with baseline model already loaded in memory
current_model = model_online

for pos in tqdm(test_positions, desc="Daily online CAB"):

    current_date = sample_dates_series.iloc[pos]

    # newly matured labeled sample available at time t:
    # origin = t - h
    update_pos = pos - CFG.h

    did_update = False
    updated_origin_date = None

    if update_pos > train_last_pos:
        # Grab a rolling window of recent history (Replay Buffer)
        start_update = max(0, update_pos - REPLAY_WINDOW + 1)
        X_update = X_all[start_update : update_pos + 1]
        Y_update = Y_all[start_update : update_pos + 1]

        X_update_scaled = transform_cov_sequences_from_params(
            X_update,
            scaler_mean_matrix,
            scaler_std_matrix
        )

        current_model = online_update_step(
            model=current_model,
            X_update_scaled=X_update_scaled,
            Y_update_unscaled=Y_update.astype(np.float32),
            device=DEVICE,
            lr=ONLINE_LR,
            epochs=ONLINE_EPOCHS,
            batch_size=ONLINE_BATCH_SIZE,
            weight_decay=0.0
        )

        did_update = True
        updated_origin_date = sample_dates_series.iloc[update_pos]

    # forecast current origin date
    X_fore = X_all[pos:pos+1]
    X_fore_scaled = transform_cov_sequences_from_params(
        X_fore,
        scaler_mean_matrix,
        scaler_std_matrix
    )

    with torch.no_grad():
        x_input = torch.tensor(X_fore_scaled, dtype=torch.float32, device=DEVICE)
        out = current_model(x_input)
        y_pred = out["y_final"][0].cpu().numpy()

    pred_list.append(y_pred)
    forecast_dates.append(current_date)

    update_log.append({
        "forecast_date": current_date,
        "did_update": did_update,
        "updated_origin_date": updated_origin_date
    })

Y_pred_online = np.stack(pred_list, axis=0).astype(np.float32)
update_log_df = pd.DataFrame(update_log)

print("Done.")
print("Y_pred_online_shape:", Y_pred_online.shape)
print(update_log_df.head(30))

In [ ]:
print(update_log_df["did_update"].value_counts(dropna=False))

first_update_rows = update_log_df.loc[update_log_df["did_update"]].head(10)
print(first_update_rows)

In [ ]:
import os
import pandas as pd

asset_order = list(CFG.asset_order)

cov_columns = []
for a in asset_order:
    for b in asset_order:
        cov_columns.append(f"cov_{a}__{b}")

rows = []

for t in range(len(forecast_dates)):
    row = {"Date": forecast_dates[t]}
    mat = Y_pred_online[t]

    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            row[f"cov_{a}__{b}"] = mat[i, j]

    rows.append(row)

cab_online_df = pd.DataFrame(rows)
cab_online_df = cab_online_df[["Date"] + cov_columns]

os.makedirs("../results", exist_ok=True)

output_path = f"../results/cab_cov_forecast_h{CFG.h}.csv"
cab_online_df.to_csv(output_path, index=False)

print("Saved online CAB forecasts to:")
print(output_path)
print(cab_online_df.head())

Testing 

In [ ]:
import pandas as pd

forecast = pd.read_csv(
    f"../results/cab_cov_forecast_h{CFG.h}.csv",
    parse_dates=["Date"]
).sort_values("Date").reset_index(drop=True)

proxy = pd.read_csv(
    f"../proxy/realized_cov_h{CFG.h}.csv",
    parse_dates=["Date"]
).sort_values("Date").reset_index(drop=True)

# covariance columns only
forecast_cov_cols = [c for c in forecast.columns if c.startswith("cov_")]
proxy_cov_cols = [c for c in proxy.columns if c.startswith("cov_")]

common_cov_cols = [c for c in forecast_cov_cols if c in proxy_cov_cols]

merged = forecast[["Date"] + common_cov_cols].merge(
    proxy[["Date"] + common_cov_cols],
    on="Date",
    how="inner",
    suffixes=("_forecast", "_proxy")
).sort_values("Date").reset_index(drop=True)

import plotly.graph_objects as go

for col in common_cov_cols:
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=merged["Date"],
            y=merged[f"{col}_forecast"],
            mode="lines",
            name="CAB forecast"
        )
    )

    fig.add_trace(
        go.Scatter(
            x=merged["Date"],
            y=merged[f"{col}_proxy"],
            mode="lines",
            name="Proxy"
        )
    )

    fig.update_layout(
        title=col,
        xaxis_title="Date",
        yaxis_title="Covariance",
        template="plotly_white",
        hovermode="x unified"
    )

    fig.show()

Optuna grid search

In [26]:
import copy
import random
from dataclasses import replace, is_dataclass

import numpy as np
import pandas as pd
import optuna
import torch
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

# =========================================================
# FALLBACK HELPERS
# =========================================================
def row_to_cov_matrix(row: pd.Series, prefix: str, asset_order) -> np.ndarray:
    """
    Build an N x N covariance matrix from columns like:
    {prefix}_cov_ASSET1__ASSET2
    """
    n = len(asset_order)
    M = np.zeros((n, n), dtype=np.float32)

    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            col = f"{prefix}_cov_{a}__{b}"
            if col not in row.index:
                raise ValueError(f"Missing column: {col}")
            M[i, j] = float(row[col])

    return M


def transform_cov_sequences_from_params(
    X: np.ndarray,
    mean_matrix: np.ndarray,
    std_matrix: np.ndarray
) -> np.ndarray:
    """
    Apply already-fitted input scaling using matrix-shaped mean/std parameters.
    X shape: (n_samples, L, N, N)
    """
    mean_matrix = np.asarray(mean_matrix, dtype=np.float32)
    std_matrix = np.asarray(std_matrix, dtype=np.float32)
    denom = np.where(np.abs(std_matrix) < 1e-12, 1.0, std_matrix)

    return ((X - mean_matrix[None, None, :, :]) / denom[None, None, :, :]).astype(np.float32)


def make_cfg_with_updates(base_cfg, **updates):
    """
    Safely clone/update CABConfig whether it is a dataclass or a plain class.
    """
    if is_dataclass(base_cfg):
        return replace(base_cfg, **updates)

    cfg = copy.deepcopy(base_cfg)
    for k, v in updates.items():
        setattr(cfg, k, v)
    return cfg


def make_spd(matrix, min_eig=1e-8, jitter=1e-12):
    matrix = np.asarray(matrix, dtype=float)

    # symmetrise
    matrix = 0.5 * (matrix + matrix.T)

    # eigenvalue clipping
    eigvals, eigvecs = np.linalg.eigh(matrix)
    eigvals = np.maximum(eigvals, min_eig)

    spd = eigvecs @ np.diag(eigvals) @ eigvecs.T

    # symmetrise again + small jitter
    spd = 0.5 * (spd + spd.T)
    spd += np.eye(spd.shape[0]) * jitter

    return spd


def frobenius_distance(A, B):
    return float(np.linalg.norm(np.asarray(A, dtype=float) - np.asarray(B, dtype=float), ord="fro"))


def qlike_loss(S_true, H_pred, eps=1e-12):
    """
    Matrix QLIKE:
        tr(S_true @ H_pred^{-1}) - logdet(S_true @ H_pred^{-1}) - n

    Computed in a numerically stable way as:
        tr(H_pred^{-1} S_true) - logdet(S_true) + logdet(H_pred) - n
    """
    S_true = make_spd(S_true, min_eig=eps, jitter=eps)
    H_pred = make_spd(H_pred, min_eig=eps, jitter=eps)

    n = S_true.shape[0]

    sign_s, logdet_s = np.linalg.slogdet(S_true)
    sign_h, logdet_h = np.linalg.slogdet(H_pred)

    if sign_s <= 0 or sign_h <= 0:
        return np.nan

    try:
        trace_term = np.trace(np.linalg.solve(H_pred, S_true))
    except np.linalg.LinAlgError:
        return np.nan

    return float(trace_term - logdet_s + logdet_h - n)


# =========================================================
# SETTINGS
# =========================================================
TARGET_H = 10
VALIDATION_START = "2020-01-01"

if TARGET_H == 10:
    VALIDATION_END = "2020-12-17"
elif TARGET_H == 21:
    VALIDATION_END = "2020-12-02"
elif TARGET_H == 63:
    VALIDATION_END = "2020-10-02"
else:
    raise ValueError("TARGET_H must be one of {10, 21, 63}")

RANDOM_STATE = 42
N_TRIALS = 50
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# fixed online-update settings to match final online evaluation
ONLINE_LR = 5e-5
ONLINE_EPOCHS = 1
ONLINE_BATCH_SIZE = 16
REPLAY_WINDOW = 1
ONLINE_WEIGHT_DECAY = 0.0

# covariance proxy file used for QLIKE / Frobenius evaluation
PROXY_PATH = f"../proxy/realized_cov_h{TARGET_H}.csv"
COV_DATA_PATH = f"../proxy/realized_cov_target_h{TARGET_H}_lag_5_10_21_63.csv"


# =========================================================
# REPRODUCIBILITY
# =========================================================
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_all_seeds(RANDOM_STATE)


# =========================================================
# TRAIN / UPDATE WRAPPERS WITH OPTIMIZER CHOICE
# =========================================================
def fit_cab_model_optuna(
    X_train_scaled,
    Y_train_unscaled,
    cfg,
    device,
    scaler_mean_matrix,
    scaler_std_matrix,
    optimizer_name="Adam",
    X_val_scaled=None,
    Y_val_unscaled=None,
):
    """
    Same structure as fit_cab_model, but with optimizer choice.
    """
    model = CABModel(
        cfg=cfg,
        scaler_mean_matrix=scaler_mean_matrix,
        scaler_std_matrix=scaler_std_matrix
    ).to(device)

    train_loader = make_dataloader(
        X=X_train_scaled,
        Y=Y_train_unscaled,
        batch_size=cfg.batch_size,
        shuffle=True
    )

    val_loader = None
    if X_val_scaled is not None and Y_val_unscaled is not None and len(X_val_scaled) > 0:
        val_loader = make_dataloader(
            X=X_val_scaled,
            Y=Y_val_unscaled,
            batch_size=cfg.batch_size,
            shuffle=False
        )

    criterion = FrobeniusLoss()

    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=cfg.learning_rate,
            weight_decay=cfg.weight_decay
        )
    elif optimizer_name == "SGD":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=cfg.learning_rate,
            weight_decay=cfg.weight_decay
        )
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    history = {"train_loss": [], "val_loss": []}

    for _ in range(cfg.epochs_initial):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device
        )
        history["train_loss"].append(train_loss)

        if val_loader is not None:
            val_loss = evaluate_one_epoch(
                model=model,
                loader=val_loader,
                criterion=criterion,
                device=device
            )
            history["val_loss"].append(val_loss)

    model.eval()
    return model, history


def online_update_step_optuna(
    model,
    X_update_scaled,
    Y_update_unscaled,
    device,
    optimizer_name="Adam",
    lr=5e-5,
    epochs=1,
    batch_size=16,
    weight_decay=0.0,
):
    """
    Same idea as online_update_step, but supports Adam or SGD.
    """
    model.train()

    X_tensor = torch.tensor(X_update_scaled, dtype=torch.float32)
    Y_tensor = torch.tensor(Y_update_unscaled, dtype=torch.float32)

    loader = DataLoader(
        TensorDataset(X_tensor, Y_tensor),
        batch_size=batch_size,
        shuffle=False,
        drop_last=False
    )

    criterion = FrobeniusLoss()

    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )
    elif optimizer_name == "SGD":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    for _ in range(epochs):
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            out = model(x_batch)
            y_pred = out["y_final"]
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()

    model.eval()
    return model


# =========================================================
# LOAD RAW DATA
# =========================================================
base_cfg = CABConfig()
base_cfg = make_cfg_with_updates(base_cfg, h=TARGET_H, input_lag=TARGET_H)

df_cov = pd.read_csv(
    COV_DATA_PATH,
    parse_dates=["Date", "target_start", "target_end"]
).sort_values("Date").reset_index(drop=True)

asset_order = tuple(base_cfg.asset_order)
n_assets = len(asset_order)

D_list = []
S_list = []

for _, row in df_cov.iterrows():
    D_list.append(row_to_cov_matrix(row, prefix=f"lag{TARGET_H}", asset_order=asset_order))
    S_list.append(row_to_cov_matrix(row, prefix="target", asset_order=asset_order))

D_all = np.asarray(D_list, dtype=np.float32)
S_all = np.asarray(S_list, dtype=np.float32)

proxy_df = pd.read_csv(PROXY_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
proxy_df = proxy_df.set_index("Date")

expected_proxy_cols = [f"cov_{a}__{b}" for a in asset_order for b in asset_order]
missing_proxy_cols = [c for c in expected_proxy_cols if c not in proxy_df.columns]
if missing_proxy_cols:
    raise ValueError(f"Missing proxy covariance columns, e.g. {missing_proxy_cols[:5]}")


# =========================================================
# SAMPLE BUILDER
# =========================================================
def build_samples_for_L(L: int):
    """
    Rebuild sliding-window CAB samples for the chosen lookback L.
    """
    X_list = []
    Y_list = []
    sample_dates = []
    sample_target_end = []

    for end_idx in range(L - 1, len(df_cov)):
        X_seq = D_all[end_idx - L + 1 : end_idx + 1]
        Y_mat = S_all[end_idx]

        X_list.append(X_seq)
        Y_list.append(Y_mat)
        sample_dates.append(df_cov.loc[end_idx, "Date"])
        sample_target_end.append(df_cov.loc[end_idx, "target_end"])

    X_all = np.asarray(X_list, dtype=np.float32)
    Y_all = np.asarray(Y_list, dtype=np.float32)
    sample_dates = pd.to_datetime(pd.Series(sample_dates)).reset_index(drop=True)
    sample_target_end = pd.to_datetime(pd.Series(sample_target_end)).reset_index(drop=True)

    return X_all, Y_all, sample_dates, sample_target_end


# =========================================================
# OPTUNA OBJECTIVE
# =========================================================
trial_results = []

def objective(trial):
    # -----------------------------
    # Hyperparameters: narrower search
    # -----------------------------
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"])

    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128, 256])
    learning_rate = trial.suggest_categorical(
        "learning_rate",
        [2e-3, 1e-3, 5e-4, 1e-4, 5e-5, 1e-5]
    )

    L = trial.suggest_categorical("L", [20,40,60,80,100,120,250])
    kernel_size = trial.suggest_categorical("kernel_size", [3, 5, 7])
    hidden_dim = trial.suggest_categorical("hidden_dim", [32, 64, 96, 128, 192, 256])
    num_lstm_layers = trial.suggest_categorical("num_lstm_layers", [2, 3, 4, 5, 6, 7])
    num_attention_heads = trial.suggest_categorical("num_attention_heads", [2, 4, 8, 16, 32])
    shrinkage_phi = trial.suggest_categorical("shrinkage_phi", [0.00, 0.20, 0.40, 0.60, 0.80, 1.00])

    # -----------------------------
    # Fixed values instead of searching
    # -----------------------------
    weight_decay = 0.0
    epochs_initial = 100

    # fixed online-update settings aligned with final evaluation
    online_lr = ONLINE_LR
    online_update_epochs = ONLINE_EPOCHS
    online_batch_size = ONLINE_BATCH_SIZE
    online_replay_window = REPLAY_WINDOW
    online_weight_decay = ONLINE_WEIGHT_DECAY

    # -----------------------------
    # Architectural constraint
    # -----------------------------
    embed_dim = 2 * hidden_dim
    if embed_dim % num_attention_heads != 0:
        raise optuna.exceptions.TrialPruned(
            f"Invalid architecture: 2*hidden_dim={embed_dim} not divisible by "
            f"num_attention_heads={num_attention_heads}"
        )

    # -----------------------------
    # CAB samples for this L
    # -----------------------------
    X_all_local, Y_all_local, sample_dates_series, sample_target_end_series = build_samples_for_L(L)

    train_mask_full = sample_target_end_series < pd.Timestamp(VALIDATION_START)
    n_train = int(train_mask_full.sum())

    if n_train < 2:
        raise optuna.exceptions.TrialPruned("No cold-start training samples available.")

    X_train_full = X_all_local[train_mask_full.to_numpy()]
    Y_train_full = Y_all_local[train_mask_full.to_numpy()]

    n_fit = max(1, int(0.8 * n_train))

    X_fit = X_train_full[:n_fit]
    Y_fit = Y_train_full[:n_fit]

    X_val = X_train_full[n_fit:]
    Y_val = Y_train_full[n_fit:]

    input_scaler = fit_input_scaler(X_fit)
    scaler_mean_matrix, scaler_std_matrix = scaler_to_matrix_params(
        input_scaler,
        n_assets=n_assets
    )

    X_fit_scaled = transform_cov_sequences(X_fit, input_scaler)
    Y_fit_unscaled = Y_fit.astype(np.float32)

    if len(X_val) > 0:
        X_val_scaled = transform_cov_sequences(X_val, input_scaler)
        Y_val_unscaled = Y_val.astype(np.float32)
    else:
        X_val_scaled = None
        Y_val_unscaled = None

    # -----------------------------
    # Trial-specific config
    # -----------------------------
    cfg = make_cfg_with_updates(
        base_cfg,
        h=TARGET_H,
        L=L,
        batch_size=batch_size,
        learning_rate=learning_rate,
        kernel_size=kernel_size,
        hidden_dim=hidden_dim,
        num_lstm_layers=num_lstm_layers,
        num_attention_heads=num_attention_heads,
        shrinkage_phi=shrinkage_phi,
        weight_decay=weight_decay,
        epochs_initial=epochs_initial,
    )

    # -----------------------------
    # Cold-start fit
    # -----------------------------
    try:
        current_model, _ = fit_cab_model_optuna(
            X_train_scaled=X_fit_scaled,
            Y_train_unscaled=Y_fit_unscaled,
            cfg=cfg,
            device=DEVICE,
            scaler_mean_matrix=scaler_mean_matrix,
            scaler_std_matrix=scaler_std_matrix,
            optimizer_name=optimizer_name,
            X_val_scaled=X_val_scaled,
            Y_val_unscaled=Y_val_unscaled
        )
    except (ValueError, RuntimeError) as e:
        raise optuna.exceptions.TrialPruned(str(e))

    # -----------------------------
    # Validation positions
    # -----------------------------
    validation_mask = (
        (sample_dates_series >= pd.Timestamp(VALIDATION_START)) &
        (sample_dates_series <= pd.Timestamp(VALIDATION_END))
    )
    validation_positions = np.where(validation_mask.to_numpy())[0]

    if len(validation_positions) == 0:
        raise optuna.exceptions.TrialPruned("No validation forecast origins found.")

    train_last_pos = np.where(train_mask_full.to_numpy())[0].max()

    qlike_list = []
    frob_list = []
    n_forecasts = 0
    n_updates = 0

    # -----------------------------
    # Online update + validation
    # -----------------------------
    for pos in validation_positions:
        current_date = sample_dates_series.iloc[pos]
        update_pos = pos - TARGET_H

        if update_pos > train_last_pos and update_pos >= 0:
            start_upd = max(0, update_pos - online_replay_window + 1)

            X_update = X_all_local[start_upd:update_pos + 1]
            Y_update = Y_all_local[start_upd:update_pos + 1]

            X_update_scaled = transform_cov_sequences_from_params(
                X_update,
                mean_matrix=scaler_mean_matrix,
                std_matrix=scaler_std_matrix
            )

            try:
                current_model = online_update_step_optuna(
                    model=current_model,
                    X_update_scaled=X_update_scaled,
                    Y_update_unscaled=Y_update.astype(np.float32),
                    device=DEVICE,
                    optimizer_name=optimizer_name,
                    lr=online_lr,
                    epochs=online_update_epochs,
                    batch_size=online_batch_size,
                    weight_decay=online_weight_decay
                )
                n_updates += 1
            except (RuntimeError, ValueError):
                pass

        X_fore = X_all_local[pos:pos + 1]
        X_fore_scaled = transform_cov_sequences_from_params(
            X_fore,
            mean_matrix=scaler_mean_matrix,
            std_matrix=scaler_std_matrix
        )

        x_input = torch.tensor(X_fore_scaled, dtype=torch.float32, device=DEVICE)

        with torch.no_grad():
            out = current_model(x_input)
            Sigma_pred = out["y_final"].detach().cpu().numpy()[0]

        Sigma_pred = make_spd(Sigma_pred)

        if current_date not in proxy_df.index:
            continue

        S_true_row = proxy_df.loc[current_date, expected_proxy_cols]
        if isinstance(S_true_row, pd.DataFrame):
            S_true_row = S_true_row.iloc[0]

        S_true = S_true_row.to_numpy(dtype=float).reshape(n_assets, n_assets)
        S_true = make_spd(S_true)

        frob = frobenius_distance(S_true, Sigma_pred)
        qlike = qlike_loss(S_true, Sigma_pred, eps=getattr(cfg, "eps", 1e-12))

        if np.isfinite(frob) and np.isfinite(qlike):
            frob_list.append(float(frob))
            qlike_list.append(float(qlike))
            n_forecasts += 1

    avg_qlike = float(np.mean(qlike_list)) if len(qlike_list) > 0 else np.nan
    avg_frobenius = float(np.mean(frob_list)) if len(frob_list) > 0 else np.nan

    trial.set_user_attr("avg_frobenius", avg_frobenius)
    trial.set_user_attr("n_forecasts", n_forecasts)
    trial.set_user_attr("n_updates", n_updates)
    trial.set_user_attr("online_lr", online_lr)
    trial.set_user_attr("online_epochs", online_update_epochs)
    trial.set_user_attr("online_batch_size", online_batch_size)
    trial.set_user_attr("online_replay_window", online_replay_window)

    result = {
        "optimizer": optimizer_name,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "L": L,
        "kernel_size": kernel_size,
        "hidden_dim": hidden_dim,
        "num_lstm_layers": num_lstm_layers,
        "num_attention_heads": num_attention_heads,
        "shrinkage_phi": shrinkage_phi,
        "weight_decay": weight_decay,
        "epochs_initial": epochs_initial,
        "online_lr": online_lr,
        "online_update_epochs": online_update_epochs,
        "online_batch_size": online_batch_size,
        "online_replay_window": online_replay_window,
        "avg_qlike": avg_qlike,
        "avg_frobenius": avg_frobenius,
        "n_forecasts": n_forecasts,
        "n_updates": n_updates,
    }
    trial_results.append(result)

    if not np.isfinite(avg_qlike):
        return 1e12

    return float(avg_qlike)


# =========================================================
# RUN OPTUNA
# =========================================================
sampler = optuna.samplers.TPESampler(
    seed=RANDOM_STATE,
    multivariate=True
)

study = optuna.create_study(
    direction="minimize",
    sampler=sampler
)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)


# =========================================================
# COLLECT RESULTS
# =========================================================
trials_df = study.trials_dataframe().copy()

trials_df["avg_frobenius"] = [t.user_attrs.get("avg_frobenius", np.nan) for t in study.trials]
trials_df["n_forecasts"] = [t.user_attrs.get("n_forecasts", np.nan) for t in study.trials]
trials_df["n_updates"] = [t.user_attrs.get("n_updates", np.nan) for t in study.trials]
trials_df["online_lr"] = [t.user_attrs.get("online_lr", np.nan) for t in study.trials]
trials_df["online_epochs"] = [t.user_attrs.get("online_epochs", np.nan) for t in study.trials]
trials_df["online_batch_size"] = [t.user_attrs.get("online_batch_size", np.nan) for t in study.trials]
trials_df["online_replay_window"] = [t.user_attrs.get("online_replay_window", np.nan) for t in study.trials]

if "value" in trials_df.columns:
    trials_df = trials_df.rename(columns={"value": "avg_qlike"})

rename_map = {
    "params_optimizer": "optimizer",
    "params_batch_size": "batch_size",
    "params_learning_rate": "learning_rate",
    "params_L": "L",
    "params_kernel_size": "kernel_size",
    "params_hidden_dim": "hidden_dim",
    "params_num_lstm_layers": "num_lstm_layers",
    "params_num_attention_heads": "num_attention_heads",
    "params_shrinkage_phi": "shrinkage_phi",
}
trials_df = trials_df.rename(columns=rename_map)

sort_cols = ["avg_qlike", "avg_frobenius"]
existing_sort_cols = [c for c in sort_cols if c in trials_df.columns]
trials_df = trials_df.sort_values(existing_sort_cols, ascending=True).reset_index(drop=True)

print("\n=========================================================")
print("OPTUNA CAB SEARCH COMPLETE")
print("Forecast horizon:", TARGET_H)
print(f"Validation period: {VALIDATION_START} -> {VALIDATION_END}")
print("=========================================================\n")

print("Best trial:")
print("  value (avg_qlike):", study.best_value)
print("  params:", study.best_params)
print("  avg_frobenius:", study.best_trial.user_attrs.get("avg_frobenius", np.nan))
print("  n_forecasts:", study.best_trial.user_attrs.get("n_forecasts", np.nan))
print("  n_updates:", study.best_trial.user_attrs.get("n_updates", np.nan))
print("  online_lr:", study.best_trial.user_attrs.get("online_lr", np.nan))
print("  online_epochs:", study.best_trial.user_attrs.get("online_epochs", np.nan))
print("  online_batch_size:", study.best_trial.user_attrs.get("online_batch_size", np.nan))
print("  online_replay_window:", study.best_trial.user_attrs.get("online_replay_window", np.nan))

print("\nTop trials ranked by QLIKE:\n")
cols_to_show = [
    "number", "state", "avg_qlike", "avg_frobenius",
    "optimizer", "batch_size", "learning_rate", "L",
    "kernel_size", "hidden_dim", "num_lstm_layers",
    "num_attention_heads", "shrinkage_phi",
    "online_lr", "online_epochs", "online_batch_size", "online_replay_window",
    "n_forecasts", "n_updates"
]
cols_to_show = [c for c in cols_to_show if c in trials_df.columns]
print(trials_df[cols_to_show].head(20).to_string(index=False))

trials_df.to_csv(f"../results/optuna_cab_search_h{TARGET_H}.csv", index=False)

/var/folders/nd/4xr06tb54pd105tcylvvn06c0000gn/T/ipykernel_75419/2126517019.py:610: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler = optuna.samplers.TPESampler(
[I 2026-04-11 10:46:07,037] A new study created in memory with name: no-name-20d000ca-be66-4559-9e88-1d0c725a3155


  0%|          | 0/50 [00:00<?, ?it/s]

Training:   0%|          | 0/27 [00:00<?, ?it/s]

Training:   0%|          | 0/27 [00:00<?, ?it/s]

Training:   0%|          | 0/27 [00:00<?, ?it/s]

...

Training:   0%|          | 0/28 [00:00<?, ?it/s]

Training:   0%|          | 0/28 [00:00<?, ?it/s]

Training:   0%|          | 0/28 [00:00<?, ?it/s]

[W 2026-04-11 15:13:41,144] Trial 3 failed with parameters: {'optimizer': 'SGD', 'batch_size': 16, 'learning_rate': 1e-05, 'L': 80, 'kernel_size': 5, 'hidden_dim': 192, 'num_lstm_layers': 5, 'num_attention_heads': 16, 'shrinkage_phi': 0.0} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/nd/4xr06tb54pd105tcylvvn06c0000gn/T/ipykernel_75419/2126517019.py", line 466, in objective
    current_model, _ = fit_cab_model_optuna(
                       ~~~~~~~~~~~~~~~~~~~~^
        X_train_scaled=X_fit_scaled,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<7 lines>...
        Y_val_unscaled=Y_val_unscaled
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/var/folders/nd/4xr06tb54pd105tcylvvn06c0000gn/T/ipykernel_75419/2126517019.py", line 216, in fit_cab_model_optuna
    train_loss = train_one_

KeyboardInterrupt: 